# ViT — An Image is Worth 16x16 Words (2020)

_Papers · Transformers & LLMs_

**Throw away convolution: cut an image into patches, treat each patch as a token, and feed a plain Transformer encoder.**

---

This notebook is a practice scaffold for the **ViT — An Image is Worth 16x16 Words (2020)** lesson. We build the tiny ViT one piece at a time — patch counting, the model, the data, then the position-embedding ablation. Run each cell, read the note above it, then tackle the practice problems at the bottom. _Save a copy to your Drive (File → Save a copy in Drive) to edit and keep your work._

In [ ]:
# Setup — numpy / pandas / scikit-learn / matplotlib ship with Colab.
import numpy as np
import matplotlib.pyplot as plt

## First, look at the data

Before training on it, see what each example actually contains. These are **images, not table columns**. We pull a few raw samples from the MNIST dataset to see them before any transforms.

In [ ]:
import torchvision

preview = torchvision.datasets.MNIST(root="./data", train=True, download=True)
print("dataset: MNIST   samples:", len(preview))
first_image, first_label = preview[0]
print("one sample:", first_image.size, "image,  label =", first_label)
print("classes:", getattr(preview, "classes", "(digit labels 0-9)"))
samples = [preview[i] for i in range(5)]
fig, axes = plt.subplots(1, 5, figsize=(8, 2))
for ax, (image, label) in zip(axes, samples):
    ax.imshow(image, cmap="gray")
    ax.set_title(str(label))
    ax.axis("off")
plt.show()

## Reference implementation — PyTorch

### Step 1 — Count the patches before building anything

ViT cuts an image into a grid of non-overlapping square patches and treats each patch as one token. Before writing the model it helps to work the arithmetic by hand: an MNIST image is 28x28 with one channel, and a patch size of 7 gives a 4x4 grid, so 16 patch tokens. We prepend one extra **class token**, making the sequence length 17, and each flattened patch has length P*P*C = 49.


In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torchvision import datasets, transforms

torch.manual_seed(0)

# Worked example: patch count for an MNIST image.
H = 28          # image height
W = 28          # image width
C = 1           # channels (MNIST is grayscale)
P = 7           # patch size (7x7 patches)
D = 32          # token / embedding width

grid = H // P                  # 28 // 7 = 4 patches per side
N = (H * W) // (P * P)         # 784 // 49 = 16 patches (tokens)
seq_len = N + 1                # +1 for the prepended class token
flat_patch = P * P * C        # length of one flattened patch -> E is (49 x D)

print("grid       =", grid, "x", grid)   # 4 x 4
print("N patches  =", N)                  # 16
print("seq len    =", seq_len)            # 17
print("flat patch =", flat_patch)         # 49


### Step 2 — Define the tiny ViT

The model turns patches into tokens with a single `Conv2d` whose kernel and stride both equal the patch size — that cuts non-overlapping patches **and** linearly projects each one to width `D` in a single op (Eq 1). A learnable class token is prepended, learnable position embeddings are added (the line the ablation will toggle via `use_pos`), a plain pre-norm Transformer encoder mixes the tokens, and we classify from the final state of the class token (Eq 4).


In [ ]:
class TinyViT(nn.Module):
    def __init__(self, img=28, P=7, C=1, D=32, depth=3, heads=4, ff=64, classes=10, use_pos=True):
        super().__init__()
        self.use_pos = use_pos
        self.N = (img // P) ** 2                                  # number of patches

        # Conv2d(kernel=stride=P): cut non-overlapping patches AND linearly project each -> token (Eq 1).
        self.patch = nn.Conv2d(C, D, kernel_size=P, stride=P)

        self.cls = nn.Parameter(torch.zeros(1, 1, D))            # learnable class token (prepended)
        self.pos = nn.Parameter(torch.zeros(1, self.N + 1, D))   # learnable position table, N+1 rows

        # Pre-norm Transformer encoder (norm_first=True).
        layer = nn.TransformerEncoderLayer(D, heads, ff, batch_first=True, norm_first=True)
        self.encoder = nn.TransformerEncoder(layer, depth)

        self.norm = nn.LayerNorm(D)
        self.head = nn.Linear(D, classes)

    def forward(self, x):                                        # x: (B, C, H, W)
        z = self.patch(x)                                        # (B, D, H/P, W/P)
        z = z.flatten(2).transpose(1, 2)                         # (B, N, D): grid -> sequence of tokens

        cls = self.cls.expand(z.shape[0], -1, -1)                # (B, 1, D)
        z = torch.cat([cls, z], dim=1)                           # (B, N+1, D): prepend class token (Eq 1)

        if self.use_pos:
            z = z + self.pos                                     # add position embeddings (Eq 1) -- the ablated line

        z = self.encoder(z)                                      # Transformer encoder (Eqns 2-3)
        return self.head(self.norm(z[:, 0]))                     # read out class token z_L^0, classify (Eq 4)


### Step 3 — Load a small MNIST subset and helpers

To keep everything fast on CPU we train on just 3000 images and test on 1000. We also define a simple `evaluate` helper that returns test accuracy, and a `train` helper that runs Adam for a few epochs. Both are plain loops so the mechanics stay visible.


In [ ]:
# A small MNIST subset (fast on CPU).
tf = transforms.ToTensor()
train_full = datasets.MNIST(root="./data", train=True,  download=True, transform=tf)
test_full  = datasets.MNIST(root="./data", train=False, download=True, transform=tf)

train_set = torch.utils.data.Subset(train_full, range(3000))     # tiny subset for speed
test_set  = torch.utils.data.Subset(test_full,  range(1000))

train_dl = torch.utils.data.DataLoader(train_set, batch_size=128, shuffle=True)
test_dl  = torch.utils.data.DataLoader(test_set,  batch_size=256)


def evaluate(net):
    net.eval()
    correct = 0
    total = 0
    with torch.no_grad():
        for x, y in test_dl:
            preds = net(x).argmax(1)
            correct += (preds == y).sum().item()
            total += y.numel()
    return correct / total


def train(use_pos, epochs=6, lr=3e-3):
    torch.manual_seed(0)
    net = TinyViT(use_pos=use_pos)
    opt = torch.optim.Adam(net.parameters(), lr=lr)
    for ep in range(epochs):
        net.train()
        for x, y in train_dl:
            loss = F.cross_entropy(net(x), y)
            opt.zero_grad()
            loss.backward()
            opt.step()
        print(f"  epoch {ep}  test-acc {evaluate(net):.3f}")
    return evaluate(net)


### Step 4 — Run the position-embedding ablation

Now we train twice, changing exactly one thing: whether the position embeddings are added. Self-attention is permutation-invariant, so with positions OFF the encoder sees a bag of patches and loses the signal of *where* each patch sat. Patch content alone still carries a lot on MNIST, so accuracy drops rather than collapsing to chance — the gap is what the position embeddings are worth.


In [ ]:
print("\nWITH position embeddings (use_pos=True):")
acc_pos = train(use_pos=True)

print("WITHOUT position embeddings (ABLATION, use_pos=False):")
acc_no = train(use_pos=False)

print(f"\nfinal test accuracy  pos-on: {acc_pos:.3f}   pos-off: {acc_no:.3f}")
# pos-on reaches ~0.93 on this tiny subset; pos-off drops to ~0.79 -- the patch-order signal is gone.
# (Exact numbers vary by hardware/seed; this is our small run, not the paper's reported result.)


## Visualize the data & results

_On a small MNIST subset, does the tiny ViT learn, and does removing the position embeddings (ablation) lower accuracy because patch order is lost?_

### Step 1 — Rebuild a compact ViT and the data

This visualization block is self-contained so you can run it on its own. It re-declares the same `TinyViT` (identical architecture and seed) and reloads the same 3000/1000 MNIST subset. Nothing here changes the model — it just sets up an isolated experiment to record accuracy *per epoch* for the two runs.


In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torchvision import datasets, transforms

torch.manual_seed(0)


class TinyViT(nn.Module):
    def __init__(self, img=28, P=7, C=1, D=32, depth=3, heads=4, ff=64, classes=10, use_pos=True):
        super().__init__()
        self.use_pos = use_pos
        self.N = (img // P) ** 2
        self.patch = nn.Conv2d(C, D, kernel_size=P, stride=P)
        self.cls = nn.Parameter(torch.zeros(1, 1, D))
        self.pos = nn.Parameter(torch.zeros(1, self.N + 1, D))
        layer = nn.TransformerEncoderLayer(D, heads, ff, batch_first=True, norm_first=True)
        self.encoder = nn.TransformerEncoder(layer, depth)
        self.norm = nn.LayerNorm(D)
        self.head = nn.Linear(D, classes)

    def forward(self, x):
        z = self.patch(x).flatten(2).transpose(1, 2)             # (B, N, D)
        z = torch.cat([self.cls.expand(z.shape[0], -1, -1), z], dim=1)   # prepend class token
        if self.use_pos:
            z = z + self.pos                                     # the ablated line
        return self.head(self.norm(self.encoder(z)[:, 0]))       # classify class token z_L^0


tf = transforms.ToTensor()
tr = torch.utils.data.Subset(datasets.MNIST("./data", True,  download=True, transform=tf), range(3000))
te = torch.utils.data.Subset(datasets.MNIST("./data", False, download=True, transform=tf), range(1000))
tr_dl = torch.utils.data.DataLoader(tr, batch_size=128, shuffle=True)
te_dl = torch.utils.data.DataLoader(te, batch_size=256)


### Step 2 — Define an accuracy probe and a per-epoch run

`acc` measures top-1 test accuracy. `run` trains a fresh ViT (same seed each time) and records the rounded accuracy after every epoch, returning the whole history. Keeping a per-epoch list lets us see the *trajectory*, not just the final number.


In [ ]:
def acc(net):
    net.eval()
    correct = 0
    total = 0
    with torch.no_grad():
        for x, y in te_dl:
            correct += (net(x).argmax(1) == y).sum().item()
            total += y.numel()
    return correct / total


def run(use_pos, epochs=6):
    torch.manual_seed(0)
    net = TinyViT(use_pos=use_pos)
    opt = torch.optim.Adam(net.parameters(), lr=3e-3)
    hist = []
    for _ in range(epochs):
        net.train()
        for x, y in tr_dl:
            loss = F.cross_entropy(net(x), y)
            opt.zero_grad()
            loss.backward()
            opt.step()
        hist.append(round(acc(net), 3))
    return hist


### Step 3 — Compare the two accuracy curves

With positions ON, accuracy climbs toward ~0.93. With positions OFF it plateaus around ~0.79: no position signal means patch order is lost. Printing both histories side by side makes the gap easy to read.


In [ ]:
print("pos on :", run(True))
print("pos off:", run(False))
# pos on -> climbs to ~0.93. pos off -> plateaus ~0.79 (no position signal; patch order lost).


## Practice

Try each one in the empty cell below it, then reveal the worked solution.

**Problem 1.** The ablation. Your tiny ViT classifies the MNIST subset well with position embeddings ON.
            Remove the single line that adds $E_\text{pos}$ to the tokens and retrain. What happens to accuracy,
            and what does that demonstrate about the patch tokens?

In [ ]:
# Your turn:


<details><summary>Show worked solution</summary>

- Delete only the position signal: change z = tokens + pos_embed to z = tokens; keep depth, width, heads, optimizer, data, and seed identical. — _An honest ablation changes exactly one thing &mdash; the position embedding &mdash; so any difference is attributable to it._
- Retrain and compare test accuracy: with $E_\text{pos}$ ON it is higher; with it OFF it drops (in our run from ~0.93 to ~0.79). — _Self-attention is permutation-invariant, so without position the encoder sees a bag of patches and cannot use where a stroke sits &mdash; yet patch content alone still carries a lot of signal on MNIST, so it does not collapse to chance._
- Conclude that position embeddings, not extra capacity, supply the spatial information; the gap is what they are worth. — _Both runs share architecture and parameter count; only the $+E_\text{pos}$ differs, isolating it as the cause._

**Answer:** With position embeddings removed, test accuracy drops (in our run ~0.93 &rarr; ~0.79). It does
                 not fall to chance because each patch token still carries its own pixels, and on MNIST the
                 content of patches is itself informative. But the model loses the ability to use where each
                 patch sat &mdash; self-attention is permutation-invariant &mdash; so it can no longer distinguish
                 patterns that differ only by spatial arrangement. Since the two runs are identical except for the
                 "$+E_\text{pos}$", this isolates the position embeddings as the source of spatial information. The
                 CODEVIZ panel shows the gap.

</details>

**Problem 2.** A colleague wants to feed a $32\times32$ RGB image ($C=3$) to your ViT with patch size $P=8$. How many
            patch tokens does the encoder see, how long is the sequence including the class token, what length is
            each flattened patch, and what shape must $E$ and $E_\text{pos}$ be (use $D=64$)?

In [ ]:
# Your turn:


<details><summary>Show worked solution</summary>

- Patch count: $N = HW/P^2 = (32\times 32)/8^2 = 1024/64 = 16$. — _A $4\times4$ grid of $8\times8$ patches tiles a $32\times32$ image (Eq 1; derived in mod-vit)._
- Sequence length with the class token: $N+1 = 17$. — _One extra slot is prepended for $x_\text{class}$, whose final state is read out (Eq 4)._
- Flattened patch length: $P^2 C = 8^2 \times 3 = 192$. So $E$ is $192\times 64$ and $E_\text{pos}$ is $(N{+}1)\times D = 17\times 64$. — _$E$ projects each flattened patch to width $D=64$; $E_\text{pos}$ has one learnable row per sequence position._

**Answer:** $N = 1024/64 = 16$ patch tokens, so the encoder sees a length-$17$ sequence (16 patches + 1 class
                 token). Each flattened patch is $P^2C = 192$ long, so $E$ is $192\times 64$ and the position table
                 $E_\text{pos}$ is $17\times 64$. Note RGB ($C=3$) only changes the flattened-patch length
                 ($P^2C$), not the patch count $N$.

</details>

**Problem 3.** The paper says ViT underperforms CNNs when trained on small datasets but matches or beats them
            after large-scale pre-training (&sect;4.3). Explain why, in terms of inductive bias, and what that
            implies for your tiny MNIST run.

In [ ]:
# Your turn:


<details><summary>Show worked solution</summary>

- Name the CNN's built-in assumptions: locality (nearby pixels) and translation equivariance (a shifted object is the same object). — _These inductive biases let a CNN learn from fewer examples because the right structure is pre-wired (&sect;3.1)._
- Note ViT lacks them: its self-attention is global from layer one and it must learn spatial structure (including position) from data. — _More freedom means more to learn, so ViT is data-hungry &mdash; "Transformers lack some of the inductive biases inherent to CNNs" (&sect;4.3)._
- Conclude that with little data CNNs win, but at $14$M&ndash;$300$M images "large scale training trumps inductive bias" and ViT catches up or passes them. — _Enough data lets ViT learn the structure a CNN assumed, plus relationships a CNN's locality forbids._

**Answer:** CNNs bake in locality and translation equivariance, so they learn well from little data. ViT has
                 neither &mdash; its attention is global and it must learn spatial structure (and position) from
                 examples &mdash; so it is data-hungry and loses to CNNs on small datasets, but at large pre-training
                 scale "large scale training trumps inductive bias" (&sect;4.3) and it matches or beats them. For our
                 tiny MNIST run this means the goal is only to show ViT learns and that position embeddings
                 help &mdash; not to beat a CNN, which it would not on data this small.

</details>